In [ ]:
# 03b_train_ensemble.ipynb
# ✨ v3.8.0: 동적 모델 조합 지원
#   - active_model: 'lgbm+rf', 'lgbm+rf+mlp' 등 '+' 구분자로 앙상블 지정
#   - 가중치 최적화: SLSQP (n변수, sum=1 등식 제약) — 모델 수 무관 단일 방법
#   - 폴더명: short 약칭 조합 (예: lgbm+rf)
# ✨ v3.10.0:
#   - log_return_1d 타겟 인식 버그 수정: target_prefix를 target_type에 따라 동적 결정
#   - base_canonical 취약점 수정: date/ticker/fold/true_ 컬럼만으로 명시적 DataFrame 구성
#   - 사다리꼴 적분 로직을 trapezoid_log_close() 모듈로 교체

import pandas as pd
import numpy as np
from pathlib import Path
from scipy.optimize import minimize
from scipy.stats import spearmanr

from src.utils.config import (
    load_config,
    parse_active_model,
    get_folder_name,
    is_ensemble,
)
from src.utils.trapezoidal import trapezoid_log_close
from src.models.ensemble_model import EnsembleModel
from src.models.artifact import save_model_artifact

In [ ]:
# ==========================================
# 1. 환경 설정
# ==========================================
cfg = load_config()
ref_date    = cfg['project']['reference_date']
model_date  = cfg['universe']['model_date']
target_type = cfg['training'].get('target_type', 'log_close')
active_model_str = cfg.get('active_model', 'lgbm+rf')

if not is_ensemble(active_model_str):
    raise ValueError(
        f"active_model='{active_model_str}'은 단일 모델입니다.\n"
        f"03b는 앙상블 전용입니다. '+' 구분자로 조합을 지정하세요.\n"
        f"예: 'lgbm+rf', 'lgbm+rf+mlp'"
    )

# 구성 모델 파싱: [(canonical, short), ...]
model_specs = parse_active_model(active_model_str)
n_models    = len(model_specs)
folder_name = get_folder_name(active_model_str)   # 예: 'lgbm+rf'

training_base  = Path(cfg['paths']['training_dir']) / model_date
ensemble_dir   = training_base / folder_name
ensemble_dir.mkdir(parents=True, exist_ok=True)

print(f"🎯 앙상블 구성: {active_model_str}  →  폴더: {folder_name}/")
print(f"   구성 모델 수: {n_models}")
for canonical, short in model_specs:
    print(f"   - {short} ({canonical})")
print(f"   target_type: {target_type}")

In [ ]:
# ==========================================
# 2. 구성 모델 val_predictions 동적 로드
# ==========================================

# ✨ v3.10.0: target_type에 따라 pred_ 컬럼 접두어를 동적으로 결정
# 구버전(v3.9.x 이전)에서 log_return 모드를 사용하면 아래 prefix가 달라질 수 있으므로
# target_type 분기를 명시한다.
if target_type == 'log_return_1d':
    target_prefix = 'pred_target_log_return_1d_h'
elif target_type == 'log_close':
    target_prefix = 'pred_target_log_close_h'
else:
    raise ValueError(
        f"지원하지 않는 target_type: '{target_type}'.\n"
        f"config.yaml의 training.target_type을 'log_close' 또는 'log_return_1d'로 설정하세요."
    )

print(f"\n📥 검증 폴드 예측 결과 로드 중...")
print(f"   pred 컬럼 접두어: '{target_prefix}'")

val_dfs   = {}   # {canonical: DataFrame}
pred_cols = None
true_cols = None

for canonical, short in model_specs:
    path = training_base / canonical / 'val_predictions.parquet'
    df   = pd.read_parquet(path)
    val_dfs[canonical] = df
    print(f"   ✅ {short}: {path}")

    if pred_cols is None:
        pred_cols = [c for c in df.columns if c.startswith(target_prefix)]
        if not pred_cols:
            raise KeyError(
                f"'{target_prefix}*' 패턴의 컬럼이 없습니다.\n"
                f"   파일: {path}\n"
                f"   실제 컬럼(pred_*): {[c for c in df.columns if c.startswith('pred_')]}\n"
                f"   config.yaml의 target_type('{target_type}')과 03단계 학습 시 사용한 "
                f"target_type이 일치하는지 확인하세요."
            )
        true_cols = [c.replace('pred_', 'true_', 1) for c in pred_cols]
        missing   = [c for c in true_cols if c not in df.columns]
        if missing:
            raise KeyError(f"true_cols 누락: {missing}")

print(f"\n   pred_cols: {pred_cols}")
print(f"   true_cols: {true_cols}")

# 기준 DataFrame (NaN 마스크는 첫 번째 모델 기준)
base_canonical = model_specs[0][0]
trues_raw = val_dfs[base_canonical][true_cols].values
valid_mask = ~np.isnan(trues_raw).any(axis=1)
trues = trues_raw[valid_mask]

# 각 모델 예측 배열 리스트
preds_list = []
for canonical, _ in model_specs:
    arr = val_dfs[canonical][pred_cols].values[valid_mask]
    preds_list.append(arr)

print(f"\n   유효 샘플: {valid_mask.sum():,}")

In [ ]:
# ==========================================
# 3. 가중치 최적화
# ==========================================

def neg_ic(weights_arr):
    """가중 혼합 예측의 -IC(Spearman) 반환. 최소화 목표."""
    blended = sum(w * p for w, p in zip(weights_arr, preds_list))
    flat_b  = np.ravel(blended)
    flat_t  = np.ravel(trues)
    mask    = ~np.isnan(flat_b) & ~np.isnan(flat_t)
    if mask.sum() < 2:
        return 0.0
    ic, _ = spearmanr(flat_b[mask], flat_t[mask])
    return float(-ic) if not np.isnan(ic) else 0.0


# SLSQP: 모델 수에 관계없이 적용 가능한 최적화 방법(method)
# - sum=1 등식 제약으로 탐색 공간이 대칭적
# - 입력 순서('rf+mlp' vs 'mlp+rf')와 무관하게 동일한 최적 IC 산출
print(f"⚙️  {n_models}-model 최적화: SLSQP (sum=1 등식 제약)")

constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}
result = minimize(
    neg_ic,
    x0=[1.0 / n_models] * n_models,
    bounds=[(0.0, 1.0)] * n_models,
    method='SLSQP',
    constraints=constraints
)
optimal_weights = [float(w) for w in result.x]

print(f"\n✅ 최적 가중치 (검증셋 기준)")
for (canonical, short), w in zip(model_specs, optimal_weights):
    print(f"   {short:<6}: {w:.4f}")
print(f"   Maximized IC: {-result.fun:.6f}")

In [ ]:
# ==========================================
# 4. 테스트셋 최종 성능 확인 (가중치 변경 없음)
# ==========================================
print("\n📊 테스트셋 앙상블 성능 확인 (RMSE & IC)...")

test_dfs   = {}
for canonical, short in model_specs:
    path = training_base / canonical / 'test_predictions.parquet'
    test_dfs[canonical] = pd.read_parquet(path)

trues_test_raw  = test_dfs[base_canonical][true_cols].values
valid_test_mask = ~np.isnan(trues_test_raw).any(axis=1)
trues_test      = trues_test_raw[valid_test_mask]
preds_test_list = [
    test_dfs[canonical][pred_cols].values[valid_test_mask]
    for canonical, _ in model_specs
]

blended_test = sum(w * p for w, p in zip(optimal_weights, preds_test_list))

def get_metrics(y_true, y_pred):
    rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))
    ic, _ = spearmanr(y_pred.ravel(), y_true.ravel())
    return rmse, float(ic)

ens_rmse, ens_ic = get_metrics(trues_test, blended_test)

print(f"\n{'Model':<14} | {'RMSE':>10} | {'IC':>8}")
print("-" * 38)
print(f"{'Ensemble':<14} | {ens_rmse:>10.6f} | {ens_ic:>8.4f}")

best_single_ic = -np.inf
for (canonical, short), preds_t in zip(model_specs, preds_test_list):
    rmse, ic = get_metrics(trues_test, preds_t)
    print(f"{short:<14} | {rmse:>10.6f} | {ic:>8.4f}")
    best_single_ic = max(best_single_ic, ic)

print(f"\n💡 단일 모델 최고 IC 대비 앙상블 개선: {ens_ic - best_single_ic:+.4f}")

In [ ]:
# ==========================================
# 5. EnsembleModel 조립 및 아티팩트 저장
# ==========================================
print("\n📦 개별 모델 로드 및 앙상블 조립 중...")

def _load_model(canonical: str, model_dir: Path):
    pkl_files = list(model_dir.glob('*.pkl'))
    if not pkl_files:
        raise FileNotFoundError(f"모델 파일 없음: {model_dir}")
    path = str(pkl_files[0])

    if canonical == 'lightgbm':
        from src.models.lightgbm_model import LightGBMModel
        return LightGBMModel.load(path)
    elif canonical == 'randomforest':
        from src.models.randomforest_model import RandomForestMultiModel
        return RandomForestMultiModel.load(path)
    elif canonical == 'mlp':
        from src.models.mlp_model import MLPModel
        return MLPModel.load(path)
    else:
        raise ValueError(f"알 수 없는 모델: {canonical}")

loaded_models = []
for canonical, short in model_specs:
    m = _load_model(canonical, training_base / canonical)
    loaded_models.append(m)
    print(f"   ✅ {short} 로드 완료")

ensemble = EnsembleModel(
    model_version=f"v1_ens_{ref_date}",
    models=loaded_models,
    weights=optimal_weights
)

weights_meta = {short: w for (_, short), w in zip(model_specs, optimal_weights)}

save_model_artifact(
    model_name=folder_name,
    model_version=ensemble.model_version,
    model_object=ensemble,
    metadata={
        'composition': [canonical for canonical, _ in model_specs],
        'weights': weights_meta,
        'val_ic': -result.fun,
        'test_ic': ens_ic,
        'target_columns': ensemble.target_columns,
    },
    model_dir=ensemble_dir
)
print(f"\n✅ 앙상블 모델 저장 완료: {ensemble_dir}/")

In [ ]:
# ==========================================
# 6. 앙상블 val/test predictions 저장
# ==========================================
# ✨ v3.10.0: base_canonical 취약점 수정
#   수정 전: base_canonical DataFrame을 통째로 copy() 후 pred_ 컬럼만 덮어쓰는 방식.
#            → 첫 번째 모델의 메타 컬럼이 암묵적으로 포함되어 구성 모델 순서가 달라지면
#              05단계에서 잘못된 메타값(fold, close 등)을 조용히 참조할 수 있었음.
#   수정 후: date, ticker, fold, true_ 컬럼만 명시적으로 추출해 새 DataFrame을 구성하고
#            pred_ 컬럼을 직접 채움.
print("\n📊 앙상블 예측 결과 저장 중...")

# ── 공통 헬퍼: 명시적 DataFrame 구성 ──────────────────────────────────────
def _build_ensemble_df(
    base_df: pd.DataFrame,
    mask: np.ndarray,
    blended: np.ndarray,
    pred_cols: list,
    true_cols: list,
) -> pd.DataFrame:
    """
    date, ticker, fold, true_ 컬럼만 가져와 새 DataFrame을 구성하고
    앙상블 혼합 pred_ 컬럼을 주입합니다.
    base_canonical의 기타 메타 컬럼(close, target_* 등)은 포함하지 않습니다.
    """
    keep_cols = ['date', 'ticker', 'fold'] + true_cols
    # fold 컬럼이 없는 경우 대비
    keep_cols = [c for c in keep_cols if c in base_df.columns]
    result = base_df[keep_cols].iloc[mask].copy().reset_index(drop=True)
    for i, col in enumerate(pred_cols):
        result[col] = blended[:, i]
    return result

# ── val_predictions ────────────────────────────────────────────────────────
blended_val = sum(w * p for w, p in zip(optimal_weights, preds_list))
df_ens_val  = _build_ensemble_df(
    val_dfs[base_canonical], valid_mask, blended_val, pred_cols, true_cols
)
val_out = ensemble_dir / 'val_predictions.parquet'
df_ens_val.to_parquet(val_out, index=False)
print(f"   ✅ val_predictions  → {val_out}")

# ── test_predictions ───────────────────────────────────────────────────────
df_ens_test = _build_ensemble_df(
    test_dfs[base_canonical], valid_test_mask, blended_test, pred_cols, true_cols
)
test_out = ensemble_dir / 'test_predictions.parquet'
df_ens_test.to_parquet(test_out, index=False)
print(f"   ✅ test_predictions → {test_out}")